# Module 08 — Notebook 3: Comparing Groups

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain what a null hypothesis is and what p < 0.05 means (and doesn't mean)
- Compute Cohen's d to measure effect size between two groups
- Write a bootstrap confidence interval function using only basic Python
- Distinguish statistical significance from practical significance

**Estimated time:** ~25 minutes

## Why This Matters for AI Research Engineering

You ran an evaluation. Model A scored 0.84 on average; Model B scored 0.79. Is that difference real, or could it just be noise from the small sample size?

This is exactly the kind of question that comes up every time you compare prompt variants, fine-tuning runs, or model versions. Without statistical tools, you might ship the "better" model based on noise — or miss a real improvement because the sample was too small to detect it.

You don't need to be a statistician. But you do need to know:
1. How to check whether a difference is likely real (t-test / p-value)
2. How big that difference actually is in practice (effect size)
3. How to estimate the uncertainty around a statistic (confidence intervals)

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_length, check_contains
import statistics
import random
import math
print("Setup complete.")

## 1. The Null Hypothesis

When comparing two groups, we start by assuming there is **no real difference** — this is called the **null hypothesis** (H₀).

For example:
- H₀: "Model A and Model B have the same true accuracy"
- H₁ (alternative): "Model A and Model B have different true accuracy"

A **p-value** tells you: "If the null hypothesis were true, how likely is it that we'd see a difference at least this large just by chance?"

- **p < 0.05**: The difference is unlikely to be chance alone (we reject H₀). Conventionally called "statistically significant".
- **p ≥ 0.05**: We don't have enough evidence to reject H₀. This does NOT mean the models are definitely the same — it might just mean we need more data.

**Important caveat:** Statistical significance ≠ practical significance. A tiny p-value with a huge dataset might reflect a difference of 0.001 — real, but meaningless in practice.

In [ ]:
# Let's see what a t-test result looks like conceptually
# We'll compute it manually below; first, let's understand the output

# Scenario: two models on the same 10 tasks
model_a = [0.85, 0.88, 0.82, 0.90, 0.87, 0.84, 0.89, 0.86, 0.83, 0.91]
model_b = [0.79, 0.75, 0.81, 0.77, 0.80, 0.76, 0.78, 0.74, 0.82, 0.79]

mean_a = statistics.mean(model_a)
mean_b = statistics.mean(model_b)

print(f"Model A mean: {mean_a:.4f}")
print(f"Model B mean: {mean_b:.4f}")
print(f"Difference:   {mean_a - mean_b:.4f}")
print()
print("Is this difference real, or could it be chance? We need a t-test to find out.")

## 2. A Simple t-Test (Welch's)

A **t-test** compares the means of two groups while accounting for their spread and sample sizes. We'll compute Welch's t-test manually — it handles groups with different variances or sizes.

The t-statistic formula:

```
t = (mean_a - mean_b) / sqrt(var_a/n_a + var_b/n_b)
```

A larger absolute t means a more significant difference. We won't compute the exact p-value from scratch (that requires a t-distribution table), but we'll look at the t-statistic as a guide:
- |t| > 2 is a rough rule of thumb for "likely significant" with moderate sample sizes
- In real research, use `scipy.stats.ttest_ind()` for the exact p-value

In [ ]:
import statistics
import math

def welch_t(group_a, group_b):
    """Compute Welch's t-statistic for two independent samples."""
    mean_a = statistics.mean(group_a)
    mean_b = statistics.mean(group_b)
    var_a  = statistics.variance(group_a)
    var_b  = statistics.variance(group_b)
    n_a    = len(group_a)
    n_b    = len(group_b)
    se     = math.sqrt(var_a / n_a + var_b / n_b)
    return (mean_a - mean_b) / se

model_a = [0.85, 0.88, 0.82, 0.90, 0.87, 0.84, 0.89, 0.86, 0.83, 0.91]
model_b = [0.79, 0.75, 0.81, 0.77, 0.80, 0.76, 0.78, 0.74, 0.82, 0.79]

t = welch_t(model_a, model_b)
print(f"Welch's t-statistic: {t:.4f}")
print(f"|t| > 2? {abs(t) > 2}  (rough rule of thumb for significance)")

# Compare with noise — two samples from the same distribution
noise_a = [0.80, 0.82, 0.79, 0.81, 0.80]
noise_b = [0.81, 0.79, 0.82, 0.80, 0.81]
t_noise = welch_t(noise_a, noise_b)
print(f"\nNoise comparison t: {t_noise:.4f}")
print(f"|t| > 2? {abs(t_noise) > 2}  (not significant — just noise)")

## 3. Effect Size: Cohen's d

Even if a difference is statistically significant, it might be too small to matter in practice. **Cohen's d** measures the practical size of the difference:

```
d = (mean_a - mean_b) / pooled_std_dev
```

Where pooled std dev is:
```
pooled_std = sqrt((var_a + var_b) / 2)
```

Interpreting Cohen's d:

| |d| | Effect size |
|---------|-------------|
| ~0.2 | Small |
| ~0.5 | Medium |
| ~0.8 | Large |

A d of 0.8 means the groups differ by 0.8 standard deviations — a meaningful practical difference.

In [ ]:
import statistics
import math

def cohens_d(group_a, group_b):
    """Compute Cohen's d effect size between two groups."""
    mean_a = statistics.mean(group_a)
    mean_b = statistics.mean(group_b)
    var_a  = statistics.variance(group_a)
    var_b  = statistics.variance(group_b)
    pooled_std = math.sqrt((var_a + var_b) / 2)
    return (mean_a - mean_b) / pooled_std

model_a = [0.85, 0.88, 0.82, 0.90, 0.87, 0.84, 0.89, 0.86, 0.83, 0.91]
model_b = [0.79, 0.75, 0.81, 0.77, 0.80, 0.76, 0.78, 0.74, 0.82, 0.79]

d = cohens_d(model_a, model_b)
print(f"Cohen's d: {d:.4f}")
print()

# Interpret
if abs(d) < 0.2:
    label = "negligible"
elif abs(d) < 0.5:
    label = "small"
elif abs(d) < 0.8:
    label = "medium"
else:
    label = "large"
print(f"Effect size: {label}")

## 4. Bootstrap Confidence Intervals

A **confidence interval** gives you a range of plausible values for a statistic, not just a single point estimate. A 95% CI means: "if we repeated this experiment many times, 95% of the computed intervals would contain the true value."

**Bootstrap** is a resampling technique that doesn't require any distributional assumptions:
1. Resample your data with replacement (same size as original)
2. Compute your statistic on the resample
3. Repeat many times (e.g., 1000)
4. The 2.5th and 97.5th percentiles of the results form your 95% CI

> **JS analogy:** It's like running a simulation 1000 times and taking the middle 95% of outcomes.

In [ ]:
import statistics
import random

def bootstrap_ci(data, stat_fn=statistics.mean, n_resamples=1000, ci=0.95, seed=42):
    """
    Compute a bootstrap confidence interval for a statistic.

    Args:
        data: list of numbers
        stat_fn: function to compute the statistic (default: mean)
        n_resamples: number of bootstrap resamples
        ci: confidence level (default: 0.95)
        seed: random seed for reproducibility

    Returns:
        (lower, upper) tuple
    """
    random.seed(seed)
    n = len(data)
    boot_stats = []

    for _ in range(n_resamples):
        resample = [random.choice(data) for _ in range(n)]
        boot_stats.append(stat_fn(resample))

    boot_stats.sort()
    alpha = 1 - ci
    lower_idx = int(alpha / 2 * n_resamples)
    upper_idx = int((1 - alpha / 2) * n_resamples)

    return (round(boot_stats[lower_idx], 4), round(boot_stats[upper_idx], 4))


scores = [0.85, 0.88, 0.82, 0.90, 0.87, 0.84, 0.89, 0.86, 0.83, 0.91]
ci_lower, ci_upper = bootstrap_ci(scores)

print(f"Mean: {statistics.mean(scores):.4f}")
print(f"95% Bootstrap CI: ({ci_lower}, {ci_upper})")
print("Interpretation: we estimate the true mean lies in this range.")

## Exercise 1 — Compute Cohen's d

Two prompt variants were evaluated on 10 tasks each. Compute Cohen's d between them. Use the formula:
```
d = (mean_a - mean_b) / sqrt((var_a + var_b) / 2)
```
Store the result in `effect_size`, rounded to 4 decimal places.

In [ ]:
import statistics
import math

prompt_v1 = [0.72, 0.75, 0.70, 0.74, 0.73, 0.76, 0.71, 0.74, 0.72, 0.75]
prompt_v2 = [0.85, 0.88, 0.82, 0.87, 0.84, 0.89, 0.83, 0.86, 0.85, 0.87]

# YOUR CODE HERE
effect_size = None  # float, rounded to 4 decimal places

In [ ]:
check_type(effect_size, float, "effect_size is a float")
check_approx(effect_size, -7.4, 0.5, "effect_size is large and negative (v2 beats v1 by a lot)")

## Exercise 2 — Interpret a P-Value

You ran a t-test comparing two models and received `p_value = 0.032`. Answer the following questions by assigning to the variables below:

- `is_significant`: bool — is the result significant at p < 0.05?
- `reject_null`: bool — should you reject the null hypothesis?
- `means_models_identical`: bool — does p < 0.05 prove the models are definitely different?

In [ ]:
p_value = 0.032

# YOUR CODE HERE
is_significant = None        # bool
reject_null = None           # bool
means_models_identical = None  # bool — does p<0.05 PROVE they are definitively different?

In [ ]:
check_equal(is_significant, True, "p=0.032 is significant at p<0.05")
check_equal(reject_null, True, "we reject the null hypothesis")
check_equal(means_models_identical, False, "p<0.05 doesn't guarantee — it just means unlikely by chance")

## Exercise 3 — Write a Bootstrap CI Function

Write a function `bootstrap_mean_ci(data, n_resamples=1000, seed=0)` that returns the 95% confidence interval for the mean as a `(lower, upper)` tuple (both rounded to 4 decimal places).

Requirements:
- Use `random.seed(seed)` for reproducibility
- Use `random.choice(data)` to resample with replacement
- Use `statistics.mean()` to compute the statistic on each resample
- Take the 2.5th and 97.5th percentile of sorted bootstrap statistics

In [ ]:
import statistics
import random

def bootstrap_mean_ci(data, n_resamples=1000, seed=0):
    """Return 95% bootstrap CI for the mean as (lower, upper) rounded to 4 decimal places."""
    # YOUR CODE HERE
    pass

# Test
test_scores = [0.82, 0.85, 0.79, 0.88, 0.81, 0.84, 0.80, 0.87, 0.83, 0.86]
ci = bootstrap_mean_ci(test_scores, seed=42)
print(f"95% CI for mean: {ci}")

In [ ]:
check_type(ci, tuple, "bootstrap_mean_ci returns a tuple")
check_length(ci, 2, "CI tuple has 2 elements")
# CI lower should be less than the mean, upper greater
mean_val = statistics.mean(test_scores)
check_equal(ci[0] < mean_val, True, "CI lower is below the mean")
check_equal(ci[1] > mean_val, True, "CI upper is above the mean")
# Check the CI is in a reasonable range
check_equal(ci[0] > 0.78, True, "CI lower is plausible")
check_equal(ci[1] < 0.90, True, "CI upper is plausible")

## Exercise 4 — Full Comparison Report

Given scores for two models, produce a comparison report as a dict with keys:
- `"mean_a"`, `"mean_b"` — means rounded to 4 decimal places
- `"cohens_d"` — effect size rounded to 4 decimal places
- `"t_stat"` — Welch's t-statistic rounded to 4 decimal places
- `"ci_a"` — 95% bootstrap CI for model_a (tuple, seed=42)

Store the result in `comparison`.

In [ ]:
import statistics
import math
import random

model_a_scores = [0.88, 0.91, 0.85, 0.93, 0.89, 0.87, 0.92, 0.90, 0.86, 0.94]
model_b_scores = [0.74, 0.71, 0.77, 0.70, 0.73, 0.76, 0.72, 0.75, 0.69, 0.74]

# YOUR CODE HERE
comparison = None  # dict with keys: mean_a, mean_b, cohens_d, t_stat, ci_a
print(comparison)

In [ ]:
from src.checks import check_keys, check_approx, check_type
check_keys(comparison, ["mean_a", "mean_b", "cohens_d", "t_stat", "ci_a"], "comparison has correct keys")
check_approx(comparison["mean_a"], 0.895, 0.001, "mean_a is correct")
check_approx(comparison["mean_b"], 0.731, 0.001, "mean_b is correct")
check_equal(comparison["cohens_d"] > 5.0, True, "cohens_d is large (very different groups)")
check_equal(comparison["t_stat"] > 10.0, True, "t_stat is large")
check_type(comparison["ci_a"], tuple, "ci_a is a tuple")

## Wrap-Up

| Concept | Formula | Python | Threshold |
|---------|---------|--------|-----------|
| Welch's t | `(mean_a - mean_b) / sqrt(var_a/n_a + var_b/n_b)` | manual | \|t\| > 2 ≈ significant |
| Cohen's d | `(mean_a - mean_b) / sqrt((var_a + var_b) / 2)` | manual | 0.2=small, 0.5=med, 0.8=large |
| Bootstrap CI | resample + percentiles | manual loop | 95% CI standard |

**Key insight:** Always report effect size alongside significance. A p-value of 0.001 with d = 0.05 is statistically significant but practically irrelevant. A p-value of 0.06 with d = 1.2 might be worth investigating more despite not crossing the threshold.

**Next:** Notebook 4 — Mini-Project: build a full stats summary pipeline on the real synthetic eval data.